In [ ]:
  !pip install pydicom

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 88.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==========================================================
# ADNI V2 HIGH-GENERALIZATION PIPELINE
# Subject-level voting + weighted loss + MixUp + fine-tuning
# Holdout-ready version
# ==========================================================

# ===============================
# CELL 1 — INSTALL / IMPORTS
# ===============================
import os, glob, zipfile, shutil, gc, random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import pydicom

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_auc_score
)

# ===============================
# CELL 2 — CONFIG
# ===============================
ZIP_DIR = "/content/drive/Othercomputers/My Laptop/MRI/T1/"
WORK_DIR = "/content/work/"
ARTIFACT_DIR = "/content/drive/MyDrive/adni/v2_artifacts/"
CKPT = ARTIFACT_DIR + "checkpoint.pt"

ADNI_MERGE = "/content/drive/MyDrive/ADNI_DATA/clean/merged_adni_raw.csv"
META_CSV   = "/content/drive/MyDrive/ADNI_DATA/references/Clinical_T1w_Imaging_Cohort_Manifest_19Apr2026.csv"

IMG_SIZE = 152
BATCH_SIZE = 16
EPOCHS_PER_ZIP = 7
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

REPLAY = []
REPLAY_MAX = 500

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(ARTIFACT_DIR, exist_ok=True)

label_map = {"CN":0, "MCI":1, "AD":2}
inv_label = {0:"CN",1:"MCI",2:"AD"}

# ===============================
# CELL 3 — UTILITIES
# ===============================
def cleanup(folder):
    shutil.rmtree(folder, ignore_errors=True)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def show_disk():
    total, used, free = shutil.disk_usage("/content")
    print("Free disk:", round(free/1e9,2), "GB")

# ===============================
# CELL 4 — LOAD TABLES
# ===============================
def load_tables():
    meta = pd.read_csv(META_CSV, low_memory=False)
    adni = pd.read_csv(ADNI_MERGE, low_memory=False)
    adni["EXAMDATE"] = pd.to_datetime(adni["EXAMDATE"], errors="coerce")
    return meta, adni

# ===============================
# CELL 5 — LABELS
# ===============================
def normalize_dx(x):
    if pd.isna(x):
        return None
    x = str(x).upper()

    if any(k in x for k in ["AD","ALZ","DEMENTIA"]):
        return "AD"
    if any(k in x for k in ["MCI","EMCI","LMCI"]):
        return "MCI"
    if any(k in x for k in ["CN","NL","NORMAL"]):
        return "CN"

    return None

def nearest_dx(ptid, adni):
    rows = adni[adni["PTID"].astype(str)==str(ptid)]

    for _, r in rows.iterrows():
        for col in ["DX","DX_bl"]:
            if col in rows.columns:
                dx = normalize_dx(r[col])
                if dx in label_map:
                    return dx
    return None

# ===============================
# CELL 6 — DICOM LOADER
# ===============================
def load_dicom_volume(folder):

    slices = []
    shapes = []

    for f in Path(folder).rglob("*.dcm"):
        try:
            d = pydicom.dcmread(str(f))
            img = d.pixel_array.astype(np.float32)

            if img.ndim != 2:
                continue

            instance = getattr(d, "InstanceNumber", 0)

            shapes.append(img.shape)
            slices.append((instance, img))

        except:
            continue

    if len(slices) == 0:
        return None

    common_shape = Counter(shapes).most_common(1)[0][0]

    clean = []
    for instance, img in slices:
        if img.shape == common_shape:
            img = (img - img.min()) / (img.max() - img.min() + 1e-5)
            clean.append((instance, img))

    if len(clean) < 10:
        return None

    clean = sorted(clean, key=lambda x: x[0])

    vol = np.stack([x[1] for x in clean], axis=0)

    return vol

# ===============================
# CELL 7 — BETTER SAGITTAL SLICES
# ===============================
def select_slices(volume, k=12):

    n = volume.shape[0]

    start = int(n * 0.40)
    end   = int(n * 0.60)

    idx = np.linspace(start, end-1, k).astype(int)

    return volume[idx]

# ===============================
# CELL 8 — BUILD SAMPLES
# returns (image,label,subject)
# ===============================
def build_samples(folder, meta, adni):

    samples = []

    for subj in Path(folder).rglob("*"):

        if not subj.is_dir():
            continue

        ptid = None

        for p in str(subj).split(os.sep):
            if "_S_" in p:
                ptid = p
                break

        if ptid is None:
            continue

        dx = nearest_dx(ptid, adni)

        if dx not in label_map:
            continue

        vol = load_dicom_volume(subj)

        if vol is None:
            continue

        chosen = select_slices(vol)

        for s in chosen:
            samples.append((s, label_map[dx], ptid))

    return samples

# ===============================
# CELL 9 — DATASET
# ===============================
class MRIDataset(Dataset):

    def __init__(self, samples):

        self.samples = samples

        self.tfm = T.Compose([
            T.ToTensor(),
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.Lambda(lambda x: x.repeat(3,1,1)),
            T.RandomHorizontalFlip(),
            T.RandomRotation(8),
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):

        x,y,sid = self.samples[i]
        x = self.tfm(x)

        return x,y,sid

# ===============================
# CELL 10 — MODEL
# ===============================
def get_model():

    model = models.efficientnet_b0(weights="DEFAULT")

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features,3)

    # stage 1 freeze
    for p in model.features.parameters():
        p.requires_grad = False

    return model.to(DEVICE)

def unfreeze_all(model):
    for p in model.features.parameters():
        p.requires_grad = True

# ===============================
# CELL 11 — MIXUP
# ===============================
def mixup_data(x, y, alpha=0.4):

    lam = np.random.beta(alpha, alpha)

    idx = torch.randperm(x.size(0)).to(x.device)

    mixed = lam*x + (1-lam)*x[idx]

    return mixed, y, y[idx], lam

def mixup_loss(criterion, pred, y1, y2, lam):
    return lam*criterion(pred,y1) + (1-lam)*criterion(pred,y2)

# ===============================
# CELL 12 — TRAIN
# ===============================
def train_one_zip(model, samples, zip_idx):

    if len(samples)==0:
        return

    labels = [y for _,y,_ in samples]
    counts = Counter(labels)

    print("Class dist:",
          {inv_label[k]:v for k,v in counts.items()})

    # weighted loss
    total = sum(counts.values())
    weights = []

    for c in range(3):
        weights.append(total / (3*counts[c]))

    class_weights = torch.tensor(weights).float().to(DEVICE)

    imbalance = max(counts.values()) / min(counts.values())
    use_mixup = imbalance > 1.5

    print("MixUp:", use_mixup)

    ds = MRIDataset(samples)

    dl = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=2
    )

    if zip_idx >= 1:
        unfreeze_all(model)
        lr = 1e-5
    else:
        lr = 1e-4

    optimizer = torch.optim.AdamW(
        filter(lambda p:p.requires_grad, model.parameters()),
        lr=lr
    )

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    model.train()

    for ep in range(EPOCHS_PER_ZIP):

        total_loss = 0

        for x,y,_ in dl:

            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()

            if use_mixup:

                xm, ya, yb, lam = mixup_data(x,y)
                out = model(xm)
                loss = mixup_loss(
                    criterion, out, ya, yb, lam
                )

            else:
                out = model(x)
                loss = criterion(out,y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print("epoch", ep+1,
              "loss",
              round(total_loss/max(len(dl),1),4))

# ===============================
# CELL 13 — SUBJECT LEVEL EVAL
# ===============================
def evaluate_subject_level(model, samples):

    ds = MRIDataset(samples)

    dl = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    model.eval()

    subj_probs = defaultdict(list)
    subj_true  = {}

    with torch.no_grad():

        for x,y,sid in dl:

            x = x.to(DEVICE)

            out = model(x)
            prob = torch.softmax(out,1).cpu().numpy()

            for i in range(len(sid)):
                subj_probs[sid[i]].append(prob[i])
                subj_true[sid[i]] = int(y[i])

    y_true=[]
    y_pred=[]
    y_prob=[]

    for sid in subj_probs:

        p = np.mean(subj_probs[sid], axis=0)

        y_prob.append(p)
        y_pred.append(np.argmax(p))
        y_true.append(subj_true[sid])

    acc = accuracy_score(y_true,y_pred)
    cm = confusion_matrix(y_true,y_pred)

    report = classification_report(
        y_true,y_pred,
        target_names=["CN","MCI","AD"],
        output_dict=True
    )

    roc = roc_auc_score(
        y_true,
        np.array(y_prob),
        multi_class="ovr"
    )

    return {
        "accuracy": acc,
        "roc_auc": roc,
        "cm": cm.tolist(),
        "report": report
    }

# ===============================
# CELL 14 — SAVE RESULTS
# ===============================
def save_results(results):

    pd.DataFrame([{
        "accuracy": results["accuracy"],
        "roc_auc": results["roc_auc"]
    }]).to_csv(
        ARTIFACT_DIR+"metrics.csv",
        index=False
    )

    pd.DataFrame(results["cm"]).to_csv(
        ARTIFACT_DIR+"confusion_matrix.csv",
        index=False
    )

    pd.DataFrame(results["report"]).transpose().to_csv(
        ARTIFACT_DIR+"classification_report.csv"
    )

# ===============================
# CELL 15 — MAIN
# ===============================
# ===============================
# CELL 15 — MAIN (EXCLUDE T1_5.zip)
# ===============================
def main():

    meta, adni = load_tables()

    model = get_model()

    # ---------------------------------------
    # Load all ZIPs then EXCLUDE holdout T1_5.zip
    # ---------------------------------------
    all_zips = sorted(glob.glob(ZIP_DIR + "*.zip"))

    HOLDOUT_NAME = "T1_5.zip"

    zips = [
        z for z in all_zips
        if os.path.basename(z) != HOLDOUT_NAME
    ]

    print("All ZIP files found :", len(all_zips))
    print("Training ZIP files  :", len(zips))
    print("Excluded holdout    :", HOLDOUT_NAME)

    for i, z in enumerate(zips):

        print("="*60)
        print("ZIP", i+1, "/", len(zips))
        print(z)

        show_disk()

        # -----------------------------------
        # Clean temp folder
        # -----------------------------------
        cleanup(WORK_DIR)
        os.makedirs(WORK_DIR, exist_ok=True)

        # -----------------------------------
        # Extract current ZIP only
        # -----------------------------------
        out = Path(WORK_DIR) / Path(z).stem
        out.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(z) as f:
            f.extractall(out)

        # -----------------------------------
        # Build training samples
        # -----------------------------------
        samples = build_samples(out, meta, adni)

        print("Samples:", len(samples))

        # -----------------------------------
        # Replay memory
        # -----------------------------------
        if REPLAY:
            add = min(len(REPLAY), 100)
            samples += random.sample(REPLAY, add)

        # -----------------------------------
        # Train
        # -----------------------------------
        train_one_zip(model, samples, i)

        # -----------------------------------
        # Save checkpoint
        # -----------------------------------
        torch.save(model.state_dict(), CKPT)

        # -----------------------------------
        # Update replay
        # -----------------------------------
        REPLAY.extend(samples)
        random.shuffle(REPLAY)
        REPLAY[:] = REPLAY[-REPLAY_MAX:]

        # -----------------------------------
        # Delete extracted data
        # -----------------------------------
        cleanup(out)

    # ======================================
    # FINAL INTERNAL VALIDATION
    # ======================================
    print("="*60)
    print("Final Validation on Replay Buffer")

    results = evaluate_subject_level(model, REPLAY)

    torch.save(
        model.state_dict(),
        ARTIFACT_DIR + "final_model.pt"
    )

    save_results(results)

    print("Accuracy:", results["accuracy"])
    print("ROC AUC :", results["roc_auc"])
    print(pd.DataFrame(results["report"]).transpose())

# ===============================
# CELL 16 — RUN
# ===============================
main()

All ZIP files found : 5
Training ZIP files  : 4
Excluded holdout    : T1_5.zip
ZIP 1 / 4
/content/drive/Othercomputers/My Laptop/MRI/T1/T1_1.zip
Free disk: 157.77 GB
Samples: 45516
Class dist: {'AD': 5352, 'CN': 16752, 'MCI': 23412}
MixUp: True
epoch 1 loss 1.0869
epoch 2 loss 1.0688
epoch 3 loss 1.0619
epoch 4 loss 1.0567
epoch 5 loss 1.0563
epoch 6 loss 1.0566
epoch 7 loss 1.0551
ZIP 2 / 4
/content/drive/Othercomputers/My Laptop/MRI/T1/T1_2.zip
Free disk: 157.97 GB
Samples: 45252
Class dist: {'AD': 5363, 'CN': 17350, 'MCI': 22639}
MixUp: True
epoch 1 loss 1.0354
epoch 2 loss 0.9764
epoch 3 loss 0.9266
epoch 4 loss 0.8754
epoch 5 loss 0.8257
epoch 6 loss 0.7923
epoch 7 loss 0.7456
ZIP 3 / 4
/content/drive/Othercomputers/My Laptop/MRI/T1/T1_3.zip
Free disk: 158.01 GB
Samples: 45144
Class dist: {'AD': 4460, 'CN': 18037, 'MCI': 22747}
MixUp: True
epoch 1 loss 0.942
epoch 2 loss 0.8375
epoch 3 loss 0.7718
epoch 4 loss 0.728
epoch 5 loss 0.6931
epoch 6 loss 0.6529
epoch 7 loss 0.6263
ZIP 4

In [ ]:
# ===============================
# HOLDOUT TEST AFTER RUNTIME DISCONNECT
# Run this in a NEW Colab session
# Uses saved final_model.pt
# Tests ONLY on T1_5.zip
# ===============================

# ==========================================================
# CELL 1 — IMPORTS
# ==========================================================
import os, zipfile, shutil, gc
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import pydicom

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    roc_auc_score
)

# ==========================================================
# CELL 2 — CONFIG
# ==========================================================
ZIP_PATH = "/content/drive/Othercomputers/My Laptop/MRI/T1/T1_5.zip"
WORK_DIR = "/content/work_holdout/"
MODEL_PATH = "/content/drive/MyDrive/adni/v2_artifacts/final_model.pt"

ADNI_MERGE = "/content/drive/MyDrive/ADNI_DATA/clean/merged_adni_raw.csv"
META_CSV   = "/content/drive/MyDrive/ADNI_DATA/references/Clinical_T1w_Imaging_Cohort_Manifest_19Apr2026.csv"

OUT_DIR = "/content/drive/MyDrive/adni/v2_holdout_results/"

IMG_SIZE = 152
BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

label_map = {"CN":0,"MCI":1,"AD":2}
inv_label = {0:"CN",1:"MCI",2:"AD"}

# ==========================================================
# CELL 3 — HELPERS
# ==========================================================
def cleanup(folder):
    shutil.rmtree(folder, ignore_errors=True)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def normalize_dx(x):
    if pd.isna(x):
        return None
    x = str(x).upper()

    if any(k in x for k in ["AD","ALZ","DEMENTIA"]):
        return "AD"
    if any(k in x for k in ["MCI","EMCI","LMCI"]):
        return "MCI"
    if any(k in x for k in ["CN","NL","NORMAL"]):
        return "CN"

    return None

def nearest_dx(ptid, adni):
    rows = adni[adni["PTID"].astype(str)==str(ptid)]

    for _, r in rows.iterrows():
        for col in ["DX","DX_bl"]:
            if col in rows.columns:
                dx = normalize_dx(r[col])
                if dx in label_map:
                    return dx
    return None

# ==========================================================
# CELL 4 — DICOM
# ==========================================================
def load_dicom_volume(folder):

    slices = []
    shapes = []

    for f in Path(folder).rglob("*.dcm"):
        try:
            d = pydicom.dcmread(str(f))
            img = d.pixel_array.astype(np.float32)

            if img.ndim != 2:
                continue

            instance = getattr(d, "InstanceNumber", 0)

            shapes.append(img.shape)
            slices.append((instance, img))

        except:
            continue

    if len(slices) == 0:
        return None

    common_shape = Counter(shapes).most_common(1)[0][0]

    clean = []

    for instance, img in slices:
        if img.shape == common_shape:
            img = (img - img.min()) / (img.max() - img.min() + 1e-5)
            clean.append((instance, img))

    if len(clean) < 10:
        return None

    clean = sorted(clean, key=lambda x:x[0])

    vol = np.stack([x[1] for x in clean], axis=0)

    return vol

def select_slices(volume, k=12):

    n = volume.shape[0]

    start = int(n * 0.40)
    end   = int(n * 0.60)

    idx = np.linspace(start, end-1, k).astype(int)

    return volume[idx]

# ==========================================================
# CELL 5 — BUILD HOLDOUT SAMPLES
# ==========================================================
def build_samples(folder, meta, adni):

    samples = []

    for subj in Path(folder).rglob("*"):

        if not subj.is_dir():
            continue

        ptid = None

        for p in str(subj).split(os.sep):
            if "_S_" in p:
                ptid = p
                break

        if ptid is None:
            continue

        dx = nearest_dx(ptid, adni)

        if dx not in label_map:
            continue

        vol = load_dicom_volume(subj)

        if vol is None:
            continue

        chosen = select_slices(vol)

        for s in chosen:
            samples.append((s, label_map[dx], ptid))

    return samples

# ==========================================================
# CELL 6 — DATASET
# ==========================================================
class MRIDataset(Dataset):

    def __init__(self, samples):

        self.samples = samples

        self.tfm = T.Compose([
            T.ToTensor(),
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.Lambda(lambda x: x.repeat(3,1,1))
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):

        x,y,sid = self.samples[i]
        x = self.tfm(x)

        return x,y,sid

# ==========================================================
# CELL 7 — MODEL
# ==========================================================
def get_model():

    model = models.efficientnet_b0(weights=None)

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features,3)

    model.load_state_dict(
        torch.load(MODEL_PATH, map_location=DEVICE)
    )

    model = model.to(DEVICE)
    model.eval()

    return model

# ==========================================================
# CELL 8 — SUBJECT LEVEL TEST
# ==========================================================
def evaluate_subject_level(model, samples):

    ds = MRIDataset(samples)

    dl = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    subj_probs = defaultdict(list)
    subj_true = {}

    with torch.no_grad():

        for x,y,sid in dl:

            x = x.to(DEVICE)

            out = model(x)

            prob = torch.softmax(out,1).cpu().numpy()

            for i in range(len(sid)):
                subj_probs[sid[i]].append(prob[i])
                subj_true[sid[i]] = int(y[i])

    y_true = []
    y_pred = []
    y_prob = []

    for sid in subj_probs:

        p = np.mean(subj_probs[sid], axis=0)

        y_prob.append(p)
        y_pred.append(np.argmax(p))
        y_true.append(subj_true[sid])

    acc = accuracy_score(y_true, y_pred)

    cm = confusion_matrix(y_true, y_pred)

    report = classification_report(
        y_true,
        y_pred,
        target_names=["CN","MCI","AD"],
        output_dict=True
    )

    roc = roc_auc_score(
        y_true,
        np.array(y_prob),
        multi_class="ovr"
    )

    return acc, roc, cm, report

# ==========================================================
# CELL 9 — RUN TEST
# ==========================================================
meta = pd.read_csv(META_CSV, low_memory=False)
adni = pd.read_csv(ADNI_MERGE, low_memory=False)

cleanup(WORK_DIR)
os.makedirs(WORK_DIR, exist_ok=True)

print("Extracting holdout ZIP...")

with zipfile.ZipFile(ZIP_PATH) as f:
    f.extractall(WORK_DIR)

print("Building samples...")

samples = build_samples(WORK_DIR, meta, adni)

print("Holdout samples:", len(samples))

model = get_model()

print("Running evaluation...")

acc, roc, cm, report = evaluate_subject_level(model, samples)

# ==========================================================
# CELL 10 — SAVE RESULTS
# ==========================================================
pd.DataFrame([{
    "accuracy": acc,
    "roc_auc": roc
}]).to_csv(OUT_DIR + "metrics.csv", index=False)

pd.DataFrame(cm).to_csv(
    OUT_DIR + "confusion_matrix.csv",
    index=False
)

pd.DataFrame(report).transpose().to_csv(
    OUT_DIR + "classification_report.csv"
)

print("="*60)
print("HOLDOUT RESULTS")
print("Accuracy:", acc)
print("ROC AUC :", roc)
print(cm)
print(pd.DataFrame(report).transpose())
print("Saved to:", OUT_DIR)

Extracting holdout ZIP...
Building samples...
Holdout samples: 44772
Running evaluation...
HOLDOUT RESULTS
Accuracy: 0.7046979865771812
ROC AUC : 0.8506108878046721
[[214  76   9]
 [ 76 253  24]
 [  9  26  58]]
              precision    recall  f1-score     support
CN             0.715719  0.715719  0.715719  299.000000
MCI            0.712676  0.716714  0.714689  353.000000
AD             0.637363  0.623656  0.630435   93.000000
accuracy       0.704698  0.704698  0.704698    0.704698
macro avg      0.688586  0.685363  0.686948  745.000000
weighted avg   0.704496  0.704698  0.704585  745.000000
Saved to: /content/drive/MyDrive/adni/v2_holdout_results/


In [ ]:
# ==========================================================
# ADNI LATE FUSION PIPELINE (FIXED FOR PROCESSED CSV)
# MRI CNN (.pt) + TABULAR XGBOOST (.joblib)
# HOLDOUT TEST = T1_5.zip
# Uses processed tabular CSV
# Drops: RID, EXAMDATE, DX_CLEAN, split
# ==========================================================

# ==========================================================
# CELL 1 — INSTALL / IMPORTS
# ==========================================================
import os, zipfile, shutil, gc
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import pydicom
import joblib

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# ==========================================================
# CELL 2 — CONFIG
# ==========================================================
ZIP_PATH = "/content/drive/Othercomputers/My Laptop/MRI/T1/T1_5.zip"

CNN_MODEL = "/content/drive/MyDrive/adni/v2_artifacts/final_model.pt"

XGB_MODEL = "/content/xgb_model_top15.joblib"

TABULAR_CSV = "/content/drive/MyDrive/artifacts/adni_full_top15_features.csv"

WORK_DIR = "/content/work/"
OUT_DIR  = "/content/drive/MyDrive/adni/fusion_results/"

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

IMG_SIZE = 152
BATCH_SIZE = 16

label_map = {"CN":0,"MCI":1,"AD":2}
inv_label = {0:"CN",1:"MCI",2:"AD"}

# ==========================================================
# CELL 3 — CLEANUP
# ==========================================================
def cleanup():
    shutil.rmtree(WORK_DIR, ignore_errors=True)
    os.makedirs(WORK_DIR, exist_ok=True)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ==========================================================
# CELL 4 — LOAD TABULAR CSV
# ==========================================================
tab = pd.read_csv(TABULAR_CSV, low_memory=False)

print("Columns found:")
print(tab.columns.tolist())

# build PTID from RID if PTID missing
if "PTID" not in tab.columns and "RID" in tab.columns:
    tab["PTID"] = tab["RID"].apply(lambda x: f"UNK_S_{int(x)}")

# normalize labels
tab["DX_CLEAN"] = tab["DX_CLEAN"].astype(str).str.upper()

tab["DX_CLEAN"] = tab["DX_CLEAN"].replace({
    "NL":"CN",
    "NORMAL":"CN",
    "EMCI":"MCI",
    "LMCI":"MCI",
    "DEMENTIA":"AD"
})

tab = tab[tab["DX_CLEAN"].isin(["CN","MCI","AD"])].copy()

print("Rows:", len(tab))
print("Has PTID:", "PTID" in tab.columns)
print("Has RID :", "RID" in tab.columns)

# ==========================================================
# CELL 5 — DICOM LOADER
# ==========================================================
def load_dicom_volume(folder):

    slices = []
    shapes = []

    for f in Path(folder).rglob("*.dcm"):

        try:
            d = pydicom.dcmread(str(f))
            img = d.pixel_array.astype(np.float32)

            if img.ndim != 2:
                continue

            instance = getattr(d, "InstanceNumber", 0)

            shapes.append(img.shape)
            slices.append((instance, img))

        except:
            continue

    if len(slices) == 0:
        return None

    common_shape = Counter(shapes).most_common(1)[0][0]

    clean = []

    for instance, img in slices:
        if img.shape == common_shape:
            img = (img - img.min()) / (img.max() - img.min() + 1e-5)
            clean.append((instance, img))

    if len(clean) < 10:
        return None

    clean = sorted(clean, key=lambda x: x[0])

    volume = np.stack([x[1] for x in clean], axis=0)

    return volume

# ==========================================================
# CELL 6 — SLICE SELECTION
# ==========================================================
def select_slices(volume, k=12):

    n = volume.shape[0]

    start = int(n * 0.40)
    end   = int(n * 0.60)

    idx = np.linspace(start, end-1, k).astype(int)

    return volume[idx]

# ==========================================================
# CELL 7 — BUILD MRI HOLDOUT SAMPLES
# ==========================================================
def build_samples(folder):

    samples = []

    for subj in Path(folder).rglob("*"):

        if not subj.is_dir():
            continue

        ptid = None

        for p in str(subj).split(os.sep):
            if "_S_" in p:
                ptid = p
                break

        if ptid is None:
            continue

        # RID extracted from MRI folder name
        try:
            rid = int(ptid.split("_")[-1])
        except:
            continue

        rows = tab[tab["RID"] == rid]

        if len(rows) == 0:
            continue

        dx = rows.iloc[0]["DX_CLEAN"]

        vol = load_dicom_volume(subj)

        if vol is None:
            continue

        chosen = select_slices(vol)

        for s in chosen:
            samples.append((s, label_map[dx], ptid))

    return samples
# ==========================================================
# CELL 8 — DATASET
# ==========================================================
class MRIDataset(Dataset):

    def __init__(self, samples):

        self.samples = samples

        self.tfm = T.Compose([
            T.ToTensor(),
            T.Resize((IMG_SIZE, IMG_SIZE)),
            T.Lambda(lambda x: x.repeat(3,1,1))
        ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):

        x,y,sid = self.samples[i]
        x = self.tfm(x)

        return x,y,sid

# ==========================================================
# CELL 9 — LOAD CNN MODEL
# ==========================================================
def get_model():

    model = models.efficientnet_b0(weights=None)

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features,3)

    model.load_state_dict(
        torch.load(CNN_MODEL, map_location=DEVICE)
    )

    model.eval()
    model.to(DEVICE)

    return model

cnn = get_model()

# ==========================================================
# CELL 10 — CNN SUBJECT PROBABILITIES
# ==========================================================
def cnn_predict_subject(samples):

    ds = MRIDataset(samples)

    dl = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    subj_probs = defaultdict(list)
    subj_true = {}

    with torch.no_grad():

        for x,y,sid in dl:

            x = x.to(DEVICE)

            out = cnn(x)

            prob = torch.softmax(out,1).cpu().numpy()

            for i in range(len(sid)):
                subj_probs[sid[i]].append(prob[i])
                subj_true[sid[i]] = int(y[i])

    final = {}

    for sid in subj_probs:
        final[sid] = np.mean(subj_probs[sid], axis=0)

    return final, subj_true

# ==========================================================
# CELL 11 — LOAD XGBOOST
# ==========================================================
xgb = joblib.load(XGB_MODEL)

# ==========================================================
# CELL 12 — XGB SUBJECT PROBS
# ==========================================================
def xgb_predict_subject(subject_ids):

    rid_map = {}

    for sid in subject_ids:
        try:
            rid_map[sid] = int(str(sid).split("_")[-1])
        except:
            pass

    rows = []

    for sid, rid in rid_map.items():

        hit = tab[tab["RID"] == rid]

        if len(hit) == 0:
            continue

        row = hit.iloc[0].copy()
        row["MRI_PTID"] = sid
        rows.append(row)

    use = pd.DataFrame(rows)

    print("Matched tabular rows:", len(use))

    drop_cols = [
        "RID",
        "PTID",
        "MRI_PTID",
        "EXAMDATE",
        "DX_CLEAN",
        "split"
    ]

    feature_cols = [
        c for c in use.columns
        if c not in drop_cols
    ]

    if hasattr(xgb, "feature_names_in_"):
        feature_cols = list(xgb.feature_names_in_)

    X = use[feature_cols].copy()
    X = X.apply(pd.to_numeric, errors="coerce").fillna(0)

    probs = xgb.predict_proba(X)

    out = {}

    for i, row in use.reset_index(drop=True).iterrows():
        out[row["MRI_PTID"]] = probs[i]

    return out

# ==========================================================
# CELL 13 — FUSION EVALUATION
# ==========================================================
def evaluate_fusion(cnn_probs, xgb_probs, y_true_map, w=0.5):

    y_true = []
    y_pred = []
    y_prob = []

    common = sorted(
        set(cnn_probs.keys()) &
        set(xgb_probs.keys())
    )

    for sid in common:

        p1 = cnn_probs[sid]
        p2 = xgb_probs[sid]

        p = w*p1 + (1-w)*p2

        y_prob.append(p)
        y_pred.append(np.argmax(p))
        y_true.append(y_true_map[sid])

    acc = accuracy_score(y_true, y_pred)

    cm = confusion_matrix(y_true, y_pred)

    report = classification_report(
        y_true,
        y_pred,
        target_names=["CN","MCI","AD"],
        output_dict=True
    )

    roc = roc_auc_score(
        y_true,
        np.array(y_prob),
        multi_class="ovr"
    )

    return {
        "accuracy": acc,
        "roc_auc": roc,
        "cm": cm,
        "report": report
    }

# ==========================================================
# CELL 14 — RUN HOLDOUT
# ==========================================================
cleanup()

print("Extracting holdout zip...")

with zipfile.ZipFile(ZIP_PATH) as f:
    f.extractall(WORK_DIR)

samples = build_samples(WORK_DIR)

print("MRI samples:", len(samples))

cnn_probs, y_true_map = cnn_predict_subject(samples)

xgb_probs = xgb_predict_subject(cnn_probs.keys())

print("Subjects matched:",
      len(set(cnn_probs.keys()) & set(xgb_probs.keys())))

# ==========================================================
# CELL 15 — SEARCH BEST WEIGHT
# ==========================================================
best_acc = 0
best_w = None
best_results = None

for w in np.arange(0.0, 1.01, 0.05):

    r = evaluate_fusion(
        cnn_probs,
        xgb_probs,
        y_true_map,
        w=w
    )

    print("CNN weight =", round(w,2),
          "ACC =", round(r["accuracy"],4))

    if r["accuracy"] > best_acc:
        best_acc = r["accuracy"]
        best_w = w
        best_results = r

# ==========================================================
# CELL 16 — FINAL RESULTS
# ==========================================================
print("="*60)
print("BEST CNN WEIGHT:", best_w)
print("BEST ACCURACY :", best_results["accuracy"])
print("ROC AUC       :", best_results["roc_auc"])
print("="*60)

print(best_results["cm"])
print("="*60)

df = pd.DataFrame(best_results["report"]).transpose()
print(df)

# save files
df.to_csv(OUT_DIR + "fusion_report.csv")

pd.DataFrame(best_results["cm"]).to_csv(
    OUT_DIR + "fusion_cm.csv",
    index=False
)

pd.DataFrame([{
    "cnn_weight": best_w,
    "accuracy": best_results["accuracy"],
    "roc_auc": best_results["roc_auc"]
}]).to_csv(
    OUT_DIR + "fusion_metrics.csv",
    index=False
)

print("Saved to:", OUT_DIR)

Columns found:
['RID', 'EXAMDATE', 'CDRSB', 'CDRSB_bl', 'FAQ', 'PTCOGBEG', 'mPACCtrailsB', 'ORIGPROT', 'LDELTOTAL_BL', 'mPACCdigit_bl', 'PTADDX', 'mPACCdigit', 'MMSE_bl', 'mPACCtrailsB_bl', 'FSVERSION_bl', 'VISDATE_ptd', 'ADAS13', 'DX_CLEAN', 'split']
Rows: 11458
Has PTID: True
Has RID : True
Extracting holdout zip...
MRI samples: 44772
Matched tabular rows: 745
Subjects matched: 745
CNN weight = 0.0 ACC = 0.4027
CNN weight = 0.05 ACC = 0.404
CNN weight = 0.1 ACC = 0.4067
CNN weight = 0.15 ACC = 0.4067
CNN weight = 0.2 ACC = 0.4107
CNN weight = 0.25 ACC = 0.4094
CNN weight = 0.3 ACC = 0.4121
CNN weight = 0.35 ACC = 0.4134
CNN weight = 0.4 ACC = 0.4174
CNN weight = 0.45 ACC = 0.4201
CNN weight = 0.5 ACC = 0.4389
CNN weight = 0.55 ACC = 0.5396
CNN weight = 0.6 ACC = 0.6013
CNN weight = 0.65 ACC = 0.6765
CNN weight = 0.7 ACC = 0.7181
CNN weight = 0.75 ACC = 0.7315
CNN weight = 0.8 ACC = 0.7262
CNN weight = 0.85 ACC = 0.7195
CNN weight = 0.9 ACC = 0.7168
CNN weight = 0.95 ACC = 0.706
CNN w

In [ ]:
# ==========================================================
# ADD THIS AS NEW FINAL CELL
# SAVE FUSED MODEL OBJECT
# ==========================================================

import joblib

# ----------------------------------------------------------
# package everything needed for inference later
# ----------------------------------------------------------
fused_model = {
    "type": "late_fusion_weighted_average",
    "cnn_model_path": CNN_MODEL,
    "xgb_model_path": XGB_MODEL,
    "cnn_weight": float(best_w),
    "xgb_weight": float(1.0 - best_w),
    "label_map": label_map,
    "inv_label": inv_label,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "notes": "Final prediction = cnn_weight * CNN_probs + xgb_weight * XGB_probs"
}

# ----------------------------------------------------------
# save
# ----------------------------------------------------------
FUSED_PATH = OUT_DIR + "fused_model.joblib"

joblib.dump(fused_model, FUSED_PATH)

print("Saved fused model:")
print(FUSED_PATH)
print()
print("CNN weight :", best_w)
print("XGB weight :", round(1.0 - best_w, 4))

Saved fused model:
/content/drive/MyDrive/adni/fusion_results/fused_model.joblib

CNN weight : 0.75
XGB weight : 0.25
